In [3]:
# %pip install -q -U transformers accelerate jiwer soundfile librosa sentencepiece

import csv
import gc
import json
import logging
import math
import os
import random
import re
import string
import unicodedata
import warnings
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import librosa
import numpy as np
import soundfile as sf
import torch
from jiwer import wer
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
logging.getLogger("transformers.generation.configuration_utils").setLevel(logging.ERROR)


@dataclass
class Config:
    model_name: str = "openai/whisper-small"
    language: str = "bulgarian"
    task: str = "transcribe"
    target_sampling_rate: int = 16000

    real_manifest_path: Path = Path("/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/all_real_bg_whisper.tsv")
    synthetic_root: Path = Path("/home/anna/python/MIPT/speach_recognition/FP/data/sintetyc_data")
    output_dir: Path = Path("/home/anna/python/MIPT/speach_recognition/FP/models/whisper_bg_finetuned")

    seed: int = 42
    validation_ratio: float = 0.08
    test_ratio: float = 0.08
    max_synthetic_multiplier: float = 2.0

    batch_size: int = 2
    eval_batch_size: int = 2
    gradient_accumulation_steps: int = 8
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    num_train_epochs: int = 10
    max_grad_norm: float = 1.0
    num_workers: int = 0

    generation_min_new_tokens: int = 192
    generation_max_new_tokens_cap: int = 448
    generation_length_margin: int = 32
    train_monitor_max_items: Optional[int] = 256
    save_every_epoch: bool = False


cfg = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)
print(asdict(cfg))

{'model_name': 'openai/whisper-small', 'language': 'bulgarian', 'task': 'transcribe', 'target_sampling_rate': 16000, 'real_manifest_path': PosixPath('/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/all_real_bg_whisper.tsv'), 'synthetic_root': PosixPath('/home/anna/python/MIPT/speach_recognition/FP/data/sintetyc_data'), 'output_dir': PosixPath('/home/anna/python/MIPT/speach_recognition/FP/models/whisper_bg_finetuned'), 'seed': 42, 'validation_ratio': 0.08, 'test_ratio': 0.08, 'max_synthetic_multiplier': 2.0, 'batch_size': 2, 'eval_batch_size': 2, 'gradient_accumulation_steps': 8, 'learning_rate': 1e-05, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'num_train_epochs': 10, 'max_grad_norm': 1.0, 'num_workers': 0, 'generation_min_new_tokens': 192, 'generation_max_new_tokens_cap': 448, 'generation_length_margin': 32, 'train_monitor_max_items': 256, 'save_every_epoch': False}


In [4]:
BG_ALLOWED_CHARS_RE = re.compile(r"[^0-9a-zа-яёщьюяъь\s]")


def normalize_transcription(text: str) -> str:
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower().replace("ё", "е")
    text = text.replace("–", " ").replace("—", " ").replace("-", " ")
    text = text.translate(str.maketrans("", "", string.punctuation + "«»“”„…"))
    text = BG_ALLOWED_CHARS_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def get_token_length_stats(
    records: Sequence[Dict[str, Any]],
    processor: WhisperProcessor,
) -> Dict[str, int]:
    token_lengths = [
        len(processor.tokenizer(record["text"], add_special_tokens=False).input_ids)
        for record in records
    ]
    if not token_lengths:
        return {"min": 0, "p95": 0, "p98": 0, "max": 0}

    return {
        "min": int(np.min(token_lengths)),
        "p95": int(np.percentile(token_lengths, 95)),
        "p98": int(np.percentile(token_lengths, 98)),
        "max": int(np.max(token_lengths)),
    }


def infer_generation_max_new_tokens(
    records: Sequence[Dict[str, Any]],
    processor: WhisperProcessor,
    floor: int,
    cap: int,
    margin: int,
) -> int:
    if not records:
        return floor

    token_stats = get_token_length_stats(records, processor)
    recommended = token_stats["p98"] + margin
    return max(floor, min(cap, recommended))


def load_real_records(manifest_path: Path) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    with manifest_path.open("r", encoding="utf-8", newline="") as file_obj:
        reader = csv.DictReader(file_obj, delimiter="\t")
        for row in reader:
            path = Path(row["path"]).expanduser()
            transcription = normalize_transcription(row["transcription"])
            if not transcription:
                continue
            records.append({
                "path": path,
                "text": transcription,
                "source": "real",
            })
    if not records:
        raise RuntimeError(f"No valid real records found in {manifest_path}")
    return records


def load_synthetic_records(synthetic_root: Path) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    metadata_files = sorted(synthetic_root.rglob("metadata*.csv"))
    for metadata_path in metadata_files:
        audio_dir = metadata_path.parent
        with metadata_path.open("r", encoding="utf-8") as file_obj:
            for raw_line in file_obj:
                line = raw_line.strip()
                if not line or "|" not in line:
                    continue
                filename, transcription = line.split("|", 1)
                transcription = normalize_transcription(transcription)
                if not transcription:
                    continue
                records.append({
                    "path": audio_dir / filename,
                    "text": transcription,
                    "source": "synthetic",
                })
    if not records:
        print("Synthetic metadata not found or empty. Training will use only real data.")
    return records


def validate_records(records: Sequence[Dict[str, Any]], split_name: str) -> None:
    if not records:
        raise RuntimeError(f"Split {split_name} is empty.")

    missing_files = [str(item["path"]) for item in records if not Path(item["path"]).exists()]
    if missing_files:
        preview = "\n".join(missing_files[:5])
        raise FileNotFoundError(f"Missing audio files in {split_name}:\n{preview}")

    empty_texts = [item for item in records if not item["text"]]
    if empty_texts:
        raise RuntimeError(f"Split {split_name} contains empty normalized transcriptions.")



def split_real_records(
    real_records: Sequence[Dict[str, Any]],
    validation_ratio: float,
    test_ratio: float,
    seed: int,
) -> tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    records = list(real_records)
    rng = random.Random(seed)
    rng.shuffle(records)

    total = len(records)
    if total < 3:
        raise RuntimeError("Need at least 3 real samples to create train/validation/test splits.")

    test_count = max(1, round(total * test_ratio))
    validation_count = max(1, round(total * validation_ratio))

    while total - test_count - validation_count < 1:
        if test_count >= validation_count and test_count > 1:
            test_count -= 1
        elif validation_count > 1:
            validation_count -= 1
        else:
            break

    train_count = total - test_count - validation_count
    if train_count < 1:
        raise RuntimeError("Not enough real samples left for training after the split.")

    train_records = records[:train_count]
    validation_records = records[train_count:train_count + validation_count]
    test_records = records[train_count + validation_count:]
    return train_records, validation_records, test_records



def sample_synthetic_records(
    synthetic_records: Sequence[Dict[str, Any]],
    real_train_size: int,
    max_synthetic_multiplier: float,
    seed: int,
) -> List[Dict[str, Any]]:
    if not synthetic_records or real_train_size <= 0 or max_synthetic_multiplier <= 0:
        return []

    rng = random.Random(seed)
    synthetic_records = list(synthetic_records)
    rng.shuffle(synthetic_records)

    max_synthetic = int(real_train_size * max_synthetic_multiplier)
    if max_synthetic <= 0:
        return []
    return synthetic_records[: min(len(synthetic_records), max_synthetic)]



def summarize_split(name: str, records: Sequence[Dict[str, Any]]) -> None:
    total = len(records)
    source_counts: Dict[str, int] = {}
    for item in records:
        source_counts[item["source"]] = source_counts.get(item["source"], 0) + 1
    print(f"{name}: total={total}, sources={source_counts}")

## Подготовка данных

Валидация и тест отделяются только от реальных записей, чтобы контроль качества отражал поведение модели на настоящем аудио. Синтетика добавляется только в train и ограничивается по объему, чтобы не подавить реальные данные.

In [5]:
class WhisperASRDataset(Dataset):
    def __init__(self, records: Sequence[Dict[str, Any]], processor: WhisperProcessor, sampling_rate: int = 16000):
        self.records = list(records)
        self.processor = processor
        self.sampling_rate = sampling_rate

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        item = self.records[index]
        audio, sampling_rate = sf.read(item["path"])
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        audio = audio.astype(np.float32)
        if sampling_rate != self.sampling_rate:
            audio = librosa.resample(audio, orig_sr=sampling_rate, target_sr=self.sampling_rate)

        features = self.processor.feature_extractor(
            audio,
            sampling_rate=self.sampling_rate,
            return_attention_mask=True,
            return_tensors="np",
        )
        labels = self.processor.tokenizer(item["text"], add_special_tokens=True).input_ids
        return {
            "input_features": features.input_features[0],
            "attention_mask": features.attention_mask[0],
            "labels": labels,
            "text": item["text"],
            "path": str(item["path"]),
            "source": item["source"],
        }


class WhisperDataCollator:
    def __init__(self, processor: WhisperProcessor):
        self.processor = processor

    def __call__(self, batch: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = torch.tensor(
            np.stack([item["input_features"] for item in batch]),
            dtype=torch.float32,
        )
        attention_mask = torch.tensor(
            np.stack([item["attention_mask"] for item in batch]),
            dtype=torch.long,
        )

        padded_labels = self.processor.tokenizer.pad(
            [{"input_ids": item["labels"]} for item in batch],
            padding=True,
            return_tensors="pt",
        )
        labels = padded_labels["input_ids"].masked_fill(padded_labels["attention_mask"].ne(1), -100)

        bos_token_id = self.processor.tokenizer.bos_token_id
        if bos_token_id is not None and torch.all(labels[:, 0] == bos_token_id):
            labels = labels[:, 1:]

        return {
            "input_features": input_features,
            "attention_mask": attention_mask,
            "labels": labels,
            "texts": [item["text"] for item in batch],
            "paths": [item["path"] for item in batch],
            "sources": [item["source"] for item in batch],
        }


@torch.no_grad()
def evaluate_loader(
    model: WhisperForConditionalGeneration,
    processor: WhisperProcessor,
    dataloader: DataLoader,
    device: torch.device,
    max_new_tokens: int,
    split_name: str,
) -> Dict[str, float]:
    model.eval()
    losses: List[float] = []
    predictions: List[str] = []
    references: List[str] = []

    for batch in tqdm(dataloader, desc=f"eval:{split_name}"):
        input_features = batch["input_features"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        autocast_context = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if device.type == "cuda"
            else nullcontext()
        )
        with autocast_context:
            outputs = model(
                input_features=input_features,
                attention_mask=attention_mask,
                labels=labels,
            )
        losses.append(float(outputs.loss.detach().cpu().item()))

        generated_ids = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            language=cfg.language,
            task=cfg.task,
        ).detach().cpu()

        label_ids = labels.detach().cpu().clone()
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

        batch_predictions = processor.tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        batch_references = processor.tokenizer.batch_decode(
            label_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        predictions.extend(normalize_transcription(text) for text in batch_predictions)
        references.extend(normalize_transcription(text) for text in batch_references)

        del input_features, attention_mask, labels, outputs, generated_ids
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return {
        "loss": sum(losses) / max(1, len(losses)),
        "wer": wer(references, predictions),
    }


@torch.no_grad()
def show_predictions(
    model: WhisperForConditionalGeneration,
    processor: WhisperProcessor,
    dataset: WhisperASRDataset,
    device: torch.device,
    max_new_tokens: int,
    sample_count: int = 5,
) -> List[Dict[str, str]]:
    model.eval()
    rows: List[Dict[str, str]] = []
    limit = min(sample_count, len(dataset))

    for index in range(limit):
        batch = WhisperDataCollator(processor)([dataset[index]])
        input_features = batch["input_features"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        generated_ids = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            language=cfg.language,
            task=cfg.task,
        ).detach().cpu()
        prediction = processor.tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        rows.append({
            "path": batch["paths"][0],
            "reference": batch["texts"][0],
            "prediction": normalize_transcription(prediction),
        })

    return rows

## Обучение и чекпоинты

До обучения делаем baseline на test. Во время fine-tuning сохраняем лучший чекпоинт по `val WER`, а в логах печатаем `train`, `val` и `test` метрики, чтобы было видно реальное движение модели.

## Запуск

Сначала выполните подготовку данных и baseline, затем отдельно запустите ячейку с `trainer.fit()`. После обучения загрузится лучший чекпоинт и снова посчитается качество на test.

In [6]:
class WhisperTrainer:
    def __init__(
        self,
        model: WhisperForConditionalGeneration,
        processor: WhisperProcessor,
        train_loader: DataLoader,
        train_monitor_loader: DataLoader,
        validation_loader: DataLoader,
        test_loader: DataLoader,
        config: Config,
        device: torch.device,
        max_new_tokens: int,
    ):
        self.model = model.to(device)
        self.model.config.use_cache = False
        self.processor = processor
        self.train_loader = train_loader
        self.train_monitor_loader = train_monitor_loader
        self.validation_loader = validation_loader
        self.test_loader = test_loader
        self.config = config
        self.device = device
        self.max_new_tokens = max_new_tokens
        self.use_amp = device.type == "cuda"
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_amp)

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay,
        )

        total_update_steps = math.ceil(len(train_loader) / config.gradient_accumulation_steps) * config.num_train_epochs
        warmup_steps = int(total_update_steps * config.warmup_ratio)
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_update_steps,
        )

        self.best_val_wer = float("inf")
        self.history: List[Dict[str, Any]] = []
        self.best_dir = config.output_dir / "best"
        self.best_dir.mkdir(parents=True, exist_ok=True)

    def clear_memory(self) -> None:
        gc.collect()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()

    def save_checkpoint(self, epoch: int, metrics: Dict[str, float]) -> None:
        self.model.save_pretrained(self.best_dir)
        self.processor.save_pretrained(self.best_dir)
        state = {
            "epoch": epoch,
            "metrics": metrics,
            "config": asdict(self.config),
            "history": self.history,
        }
        with (self.best_dir / "trainer_state.json").open("w", encoding="utf-8") as file_obj:
            json.dump(state, file_obj, ensure_ascii=False, indent=2, default=str)

    def train_one_epoch(self, epoch_index: int) -> float:
        self.model.train()
        self.optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(self.train_loader, desc=f"train:{epoch_index}")
        for step, batch in enumerate(progress, start=1):
            input_features = batch["input_features"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)
            labels = batch["labels"].to(self.device)

            autocast_context = (
                torch.autocast(device_type="cuda", dtype=torch.float16)
                if self.use_amp
                else nullcontext()
            )
            with autocast_context:
                outputs = self.model(
                    input_features=input_features,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                loss = outputs.loss / self.config.gradient_accumulation_steps

            self.scaler.scale(loss).backward()

            if step % self.config.gradient_accumulation_steps == 0 or step == len(self.train_loader):
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad(set_to_none=True)
                self.scheduler.step()

            loss_value = float(loss.detach().cpu().item()) * self.config.gradient_accumulation_steps
            running_loss += loss_value
            progress.set_postfix(loss=f"{loss_value:.4f}")

            del input_features, attention_mask, labels, outputs, loss

        self.clear_memory()
        return running_loss / max(1, len(self.train_loader))

    def fit(self) -> List[Dict[str, Any]]:
        self.config.output_dir.mkdir(parents=True, exist_ok=True)

        for epoch in range(1, self.config.num_train_epochs + 1):
            train_loss = self.train_one_epoch(epoch)
            train_metrics = evaluate_loader(
                self.model,
                self.processor,
                self.train_monitor_loader,
                self.device,
                self.max_new_tokens,
                split_name="train",
            )
            val_metrics = evaluate_loader(
                self.model,
                self.processor,
                self.validation_loader,
                self.device,
                self.max_new_tokens,
                split_name="val",
            )
            test_metrics = evaluate_loader(
                self.model,
                self.processor,
                self.test_loader,
                self.device,
                self.max_new_tokens,
                split_name="test",
            )

            epoch_record = {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_eval_loss": train_metrics["loss"],
                "train_wer": train_metrics["wer"],
                "val_loss": val_metrics["loss"],
                "val_wer": val_metrics["wer"],
                "test_loss": test_metrics["loss"],
                "test_wer": test_metrics["wer"],
            }
            self.history.append(epoch_record)

            with (self.config.output_dir / "history.json").open("w", encoding="utf-8") as file_obj:
                json.dump(self.history, file_obj, ensure_ascii=False, indent=2)

            print(
                f"epoch={epoch} "
                f"train_loss={train_loss:.4f} train_wer={train_metrics['wer']:.4f} "
                f"val_loss={val_metrics['loss']:.4f} val_wer={val_metrics['wer']:.4f} "
                f"test_loss={test_metrics['loss']:.4f} test_wer={test_metrics['wer']:.4f}"
            )

            if val_metrics["wer"] < self.best_val_wer:
                self.best_val_wer = val_metrics["wer"]
                self.save_checkpoint(epoch, epoch_record)
                print(f"Saved new best checkpoint to {self.best_dir}")

            if self.config.save_every_epoch:
                epoch_dir = self.config.output_dir / f"epoch_{epoch:02d}"
                epoch_dir.mkdir(parents=True, exist_ok=True)
                self.model.save_pretrained(epoch_dir)
                self.processor.save_pretrained(epoch_dir)

            self.clear_memory()

        return self.history

In [5]:
real_records = load_real_records(cfg.real_manifest_path)
synthetic_records = load_synthetic_records(cfg.synthetic_root)

real_train_records, validation_records, test_records = split_real_records(
    real_records,
    validation_ratio=cfg.validation_ratio,
    test_ratio=cfg.test_ratio,
    seed=cfg.seed,
)
synthetic_train_records = sample_synthetic_records(
    synthetic_records,
    real_train_size=len(real_train_records),
    max_synthetic_multiplier=cfg.max_synthetic_multiplier,
    seed=cfg.seed,
)

train_records = real_train_records + synthetic_train_records
random.Random(cfg.seed).shuffle(train_records)

for split_name, split_records in [
    ("train", train_records),
    ("validation", validation_records),
    ("test", test_records),
]:
    validate_records(split_records, split_name)
    summarize_split(split_name, split_records)

processor = WhisperProcessor.from_pretrained(cfg.model_name, language=cfg.language, task=cfg.task)
real_token_stats = get_token_length_stats(
    real_train_records + validation_records + test_records,
    processor,
)
max_new_tokens = infer_generation_max_new_tokens(
    real_train_records + validation_records + test_records,
    processor,
    floor=cfg.generation_min_new_tokens,
    cap=cfg.generation_max_new_tokens_cap,
    margin=cfg.generation_length_margin,
)
print("Real token stats:", real_token_stats)
print(f"Using max_new_tokens={max_new_tokens} for evaluation/generation")

model = WhisperForConditionalGeneration.from_pretrained(cfg.model_name)

# Remove all legacy generation fields that conflict with explicit generate() args.
for obj in (model.generation_config, model.config):
    if hasattr(obj, "forced_decoder_ids"):
        obj.forced_decoder_ids = None
    if hasattr(obj, "suppress_tokens"):
        obj.suppress_tokens = None
    if hasattr(obj, "begin_suppress_tokens"):
        obj.begin_suppress_tokens = None
    if hasattr(obj, "max_length"):
        obj.max_length = None

model.generation_config.max_new_tokens = max_new_tokens

train_dataset = WhisperASRDataset(train_records, processor, sampling_rate=cfg.target_sampling_rate)
validation_dataset = WhisperASRDataset(validation_records, processor, sampling_rate=cfg.target_sampling_rate)
test_dataset = WhisperASRDataset(test_records, processor, sampling_rate=cfg.target_sampling_rate)

if cfg.train_monitor_max_items is None or len(train_dataset) <= cfg.train_monitor_max_items:
    train_monitor_dataset = train_dataset
else:
    monitor_indices = random.Random(cfg.seed).sample(range(len(train_dataset)), cfg.train_monitor_max_items)
    train_monitor_dataset = Subset(train_dataset, monitor_indices)

collator = WhisperDataCollator(processor)
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    collate_fn=collator,
)
train_monitor_loader = DataLoader(
    train_monitor_dataset,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collator,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collator,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collator,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer = WhisperTrainer(
    model=model,
    processor=processor,
    train_loader=train_loader,
    train_monitor_loader=train_monitor_loader,
    validation_loader=validation_loader,
    test_loader=test_loader,
    config=cfg,
    device=device,
    max_new_tokens=max_new_tokens,
)

baseline_val_metrics = evaluate_loader(
    trainer.model,
    processor,
    validation_loader,
    device,
    max_new_tokens,
    split_name="baseline_val",
)
baseline_test_metrics = evaluate_loader(
    trainer.model,
    processor,
    test_loader,
    device,
    max_new_tokens,
    split_name="baseline_test",
)

print("Baseline validation before training:", baseline_val_metrics)
print("Baseline test before training:", baseline_test_metrics)

show_predictions(
    trainer.model,
    processor,
    test_dataset,
    device=device,
    max_new_tokens=max_new_tokens,
    sample_count=3,
)

train: total=21240, sources={'real': 8382, 'synthetic': 12858}
validation: total=798, sources={'real': 798}
test: total=798, sources={'real': 798}


Real token stats: {'min': 3, 'p95': 62, 'p98': 78, 'max': 238}
Using max_new_tokens=192 for evaluation/generation


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

eval:baseline_val:   0%|          | 0/399 [00:00<?, ?it/s]

eval:baseline_test:   0%|          | 0/399 [00:00<?, ?it/s]

Baseline validation before training: {'loss': 2.944304641625636, 'wer': 0.4542840713053479}
Baseline test before training: {'loss': 2.841068086468785, 'wer': 0.4373987252889705}


[{'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_test_00000_of_00001_0002299.wav',
  'reference': 'той още повече се издигаше пред очите му',
  'prediction': 'той още повече се издигаше предучително'},
 {'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_test_00000_of_00001_0001056.wav',
  'reference': 'не бойте се не бойте се сиви зайченца каза им приветливо лиса',
  'prediction': 'не бойте се не бойте се си визаичата казвим приветливо лиса'},
 {'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_validation_00000_of_00001_0000100.wav',
  'reference': 'след два часа кризата полека лека се уталожи',
  'prediction': 'следва два часа кризата полека лека се отължи'}]

In [6]:
history = trainer.fit()

best_model = WhisperForConditionalGeneration.from_pretrained(trainer.best_dir).to(device)

for obj in (best_model.generation_config, best_model.config):
    if hasattr(obj, "forced_decoder_ids"):
        obj.forced_decoder_ids = None
    if hasattr(obj, "suppress_tokens"):
        obj.suppress_tokens = None
    if hasattr(obj, "begin_suppress_tokens"):
        obj.begin_suppress_tokens = None
    if hasattr(obj, "max_length"):
        obj.max_length = None

best_model.generation_config.max_new_tokens = max_new_tokens

best_val_metrics = evaluate_loader(
    best_model,
    processor,
    validation_loader,
    device,
    max_new_tokens,
    split_name="best_val",
)
best_test_metrics = evaluate_loader(
    best_model,
    processor,
    test_loader,
    device,
    max_new_tokens,
    split_name="best_test",
)

print("Best validation:", best_val_metrics)
print("Best test:", best_test_metrics)
show_predictions(
    best_model,
    processor,
    test_dataset,
    device=device,
    max_new_tokens=max_new_tokens,
    sample_count=5,
)

train:1:   0%|          | 0/10620 [00:00<?, ?it/s]

eval:train:   0%|          | 0/128 [00:00<?, ?it/s]

eval:val:   0%|          | 0/399 [00:00<?, ?it/s]

eval:test:   0%|          | 0/399 [00:00<?, ?it/s]

epoch=1 train_loss=0.5553 train_wer=0.1328 val_loss=0.3003 val_wer=0.2492 test_loss=0.2716 test_wer=0.2334


...

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

eval:best_val:   0%|          | 0/399 [00:00<?, ?it/s]

eval:best_test:   0%|          | 0/399 [00:00<?, ?it/s]

Best validation: {'loss': 0.26414309870354674, 'wer': 0.16664749856239217}
Best test: {'loss': 0.24359981549637805, 'wer': 0.1522091390299233}


[{'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_test_00000_of_00001_0002299.wav',
  'reference': 'той още повече се издигаше пред очите му',
  'prediction': 'той още повече се издигаше пред очите му'},
 {'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_test_00000_of_00001_0001056.wav',
  'reference': 'не бойте се не бойте се сиви зайченца каза им приветливо лиса',
  'prediction': 'не бойте се не бойте се сиви зайчета каза им приветливо лиса'},
 {'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_validation_00000_of_00001_0000100.wav',
  'reference': 'след два часа кризата полека лека се уталожи',
  'prediction': 'след два часа кризата полека лека се оталожи'},
 {'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_train_00001_of_00005_0000247.wav',
  'reference': 'оригиналният законопроект е изготвен от бивш

# дообучение

In [7]:
from pathlib import Path
import json
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

base_dir = Path("/home/anna/python/MIPT/speach_recognition/FP/models/whisper_bg_finetuned")
ckpt9_dir = base_dir / "epoch_09"   # будет работать, если чекпоинт эпохи 9 реально сохранен
best_dir = base_dir / "best"

# 1) Загрузка модели/процессора
load_dir = ckpt9_dir if ckpt9_dir.exists() else best_dir
print("Loading from:", load_dir)

processor = WhisperProcessor.from_pretrained(load_dir)
model = WhisperForConditionalGeneration.from_pretrained(load_dir).to(
    torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

# 2) Очистка generation-конфига (как у тебя в ноутбуке)
for obj in (model.generation_config, model.config):
    if hasattr(obj, "forced_decoder_ids"):
        obj.forced_decoder_ids = None
    if hasattr(obj, "suppress_tokens"):
        obj.suppress_tokens = None
    if hasattr(obj, "begin_suppress_tokens"):
        obj.begin_suppress_tokens = None
    if hasattr(obj, "max_length"):
        obj.max_length = None

# 3) Если нужно именно "данные" по эпохе 9 (метрики) — читаем history.json
history_path = base_dir / "history.json"
if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
    epoch9 = next((x for x in history if x.get("epoch") == 9), None)
    print("Epoch 9 metrics:", epoch9)
else:
    print("history.json not found")

Loading from: /home/anna/python/MIPT/speach_recognition/FP/models/whisper_bg_finetuned/best


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Epoch 9 metrics: {'epoch': 9, 'train_loss': 0.0009338861917214103, 'train_eval_loss': 0.0009114403310377384, 'train_wer': 0.00026831231553528306, 'val_loss': 0.2604665485831307, 'val_wer': 0.17377803335250144, 'test_loss': 0.24021466765620594, 'test_wer': 0.15210111267149184}


In [9]:
# Continue training: fixed real-train, re-sampled synthetic each epoch.
continue_epochs = 3
continue_synth_multiplier = cfg.max_synthetic_multiplier
continue_seed_base = cfg.seed + 10_000
continue_output_dir = cfg.output_dir / "continue_random_synth"
continue_output_dir.mkdir(parents=True, exist_ok=True)
continue_best_dir = continue_output_dir / "best"
continue_best_dir.mkdir(parents=True, exist_ok=True)

# Use the already available trained model in memory.
continue_model = best_model if "best_model" in globals() else trainer.model
continue_model = continue_model.to(device)
continue_model.config.use_cache = False

for obj in (continue_model.generation_config, continue_model.config):
    if hasattr(obj, "forced_decoder_ids"):
        obj.forced_decoder_ids = None
    if hasattr(obj, "suppress_tokens"):
        obj.suppress_tokens = None
    if hasattr(obj, "begin_suppress_tokens"):
        obj.begin_suppress_tokens = None
    if hasattr(obj, "max_length"):
        obj.max_length = None
continue_model.generation_config.max_new_tokens = max_new_tokens

print("Continue model source:", "best_model" if "best_model" in globals() else "trainer.model")
print("Continue checkpoints dir:", continue_output_dir)
print("Continue epochs:", continue_epochs)

Continue model source: trainer.model
Continue checkpoints dir: /home/anna/python/MIPT/speach_recognition/FP/models/whisper_bg_finetuned/continue_random_synth
Continue epochs: 3


In [10]:
# Train 3 more epochs with fixed real data + new synthetic sample each epoch.
continue_history = []
continue_best_val_wer = float("inf")

expected_synth_size = int(len(real_train_records) * continue_synth_multiplier)
expected_train_size = len(real_train_records) + min(len(synthetic_records), max(0, expected_synth_size))
expected_batches = math.ceil(expected_train_size / cfg.batch_size)
expected_update_steps = math.ceil(expected_batches / cfg.gradient_accumulation_steps)
total_update_steps = max(1, continue_epochs * expected_update_steps)
warmup_steps = int(total_update_steps * cfg.warmup_ratio)

continue_optimizer = torch.optim.AdamW(
    continue_model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
)
continue_scheduler = get_linear_schedule_with_warmup(
    continue_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)
continue_use_amp = device.type == "cuda"
continue_scaler = torch.amp.GradScaler("cuda", enabled=continue_use_amp)

for epoch in range(1, continue_epochs + 1):
    epoch_seed = continue_seed_base + epoch
    epoch_synth_records = sample_synthetic_records(
        synthetic_records,
        real_train_size=len(real_train_records),
        max_synthetic_multiplier=continue_synth_multiplier,
        seed=epoch_seed,
    )
    epoch_train_records = list(real_train_records) + epoch_synth_records
    random.Random(epoch_seed).shuffle(epoch_train_records)

    epoch_train_dataset = WhisperASRDataset(
        epoch_train_records,
        processor,
        sampling_rate=cfg.target_sampling_rate,
    )
    epoch_train_loader = DataLoader(
        epoch_train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        collate_fn=collator,
    )

    continue_model.train()
    continue_optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    progress = tqdm(epoch_train_loader, desc=f"continue_train:{epoch}")
    for step, batch in enumerate(progress, start=1):
        input_features = batch["input_features"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        autocast_context = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if continue_use_amp
            else nullcontext()
        )
        with autocast_context:
            outputs = continue_model(
                input_features=input_features,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss / cfg.gradient_accumulation_steps

        continue_scaler.scale(loss).backward()

        if step % cfg.gradient_accumulation_steps == 0 or step == len(epoch_train_loader):
            continue_scaler.unscale_(continue_optimizer)
            torch.nn.utils.clip_grad_norm_(continue_model.parameters(), cfg.max_grad_norm)
            continue_scaler.step(continue_optimizer)
            continue_scaler.update()
            continue_optimizer.zero_grad(set_to_none=True)
            continue_scheduler.step()

        loss_value = float(loss.detach().cpu().item()) * cfg.gradient_accumulation_steps
        running_loss += loss_value
        progress.set_postfix(loss=f"{loss_value:.4f}")

        del input_features, attention_mask, labels, outputs, loss

    train_loss = running_loss / max(1, len(epoch_train_loader))

    val_metrics = evaluate_loader(
        continue_model,
        processor,
        validation_loader,
        device,
        max_new_tokens,
        split_name=f"continue_val_e{epoch}",
    )
    test_metrics = evaluate_loader(
        continue_model,
        processor,
        test_loader,
        device,
        max_new_tokens,
        split_name=f"continue_test_e{epoch}",
    )

    epoch_record = {
        "epoch": epoch,
        "seed": epoch_seed,
        "real_count": len(real_train_records),
        "synthetic_count": len(epoch_synth_records),
        "train_count": len(epoch_train_records),
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_wer": val_metrics["wer"],
        "test_loss": test_metrics["loss"],
        "test_wer": test_metrics["wer"],
    }
    continue_history.append(epoch_record)

    with (continue_output_dir / "continue_history.json").open("w", encoding="utf-8") as file_obj:
        json.dump(continue_history, file_obj, ensure_ascii=False, indent=2)

    print(
        f"continue_epoch={epoch} seed={epoch_seed} "
        f"real={len(real_train_records)} synth={len(epoch_synth_records)} "
        f"train_loss={train_loss:.4f} val_wer={val_metrics['wer']:.4f} test_wer={test_metrics['wer']:.4f}"
    )

    epoch_dir = continue_output_dir / f"epoch_{epoch:02d}"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    continue_model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)

    if val_metrics["wer"] < continue_best_val_wer:
        continue_best_val_wer = val_metrics["wer"]
        continue_model.save_pretrained(continue_best_dir)
        processor.save_pretrained(continue_best_dir)
        with (continue_best_dir / "trainer_state.json").open("w", encoding="utf-8") as file_obj:
            json.dump({"best_epoch": epoch, "metrics": epoch_record}, file_obj, ensure_ascii=False, indent=2)
        print(f"Saved new continue-best checkpoint to {continue_best_dir}")

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("Continue training finished. History path:", continue_output_dir / "continue_history.json")

continue_train:1:   0%|          | 0/10620 [00:00<?, ?it/s]

eval:continue_val_e1:   0%|          | 0/399 [00:00<?, ?it/s]

eval:continue_test_e1:   0%|          | 0/399 [00:00<?, ?it/s]

continue_epoch=1 seed=10043 real=8382 synth=12858 train_loss=0.4094 val_wer=0.2140 test_wer=0.1952


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

eval:continue_val_e3:   0%|          | 0/399 [00:00<?, ?it/s]

eval:continue_test_e3:   0%|          | 0/399 [00:00<?, ?it/s]

continue_epoch=3 seed=10045 real=8382 synth=12858 train_loss=0.0422 val_wer=0.2023 test_wer=0.1657


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Continue training finished. History path: /home/anna/python/MIPT/speach_recognition/FP/models/whisper_bg_finetuned/continue_random_synth/continue_history.json


In [11]:
more_epochs = 5
resume_model = continue_model
resume_output_dir = continue_output_dir / "more_5_epochs"
resume_output_dir.mkdir(parents=True, exist_ok=True)

for obj in (resume_model.generation_config, resume_model.config):
    if hasattr(obj, "forced_decoder_ids"):
        obj.forced_decoder_ids = None
    if hasattr(obj, "suppress_tokens"):
        obj.suppress_tokens = None
    if hasattr(obj, "begin_suppress_tokens"):
        obj.begin_suppress_tokens = None
    if hasattr(obj, "max_length"):
        obj.max_length = None

resume_model.generation_config.max_new_tokens = max_new_tokens

for epoch in range(1, more_epochs + 1):
    epoch_seed = continue_seed_base + 1000 + epoch
    epoch_synth_records = sample_synthetic_records(
        synthetic_records,
        real_train_size=len(real_train_records),
        max_synthetic_multiplier=continue_synth_multiplier,
        seed=epoch_seed,
    )
    epoch_train_records = list(real_train_records) + epoch_synth_records
    random.Random(epoch_seed).shuffle(epoch_train_records)

    epoch_train_dataset = WhisperASRDataset(epoch_train_records, processor, sampling_rate=cfg.target_sampling_rate)
    epoch_train_loader = DataLoader(
        epoch_train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        collate_fn=collator,
    )

    resume_model.train()
    continue_optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(epoch_train_loader, start=1):
        input_features = batch["input_features"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with (torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()):
            outputs = resume_model(input_features=input_features, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / cfg.gradient_accumulation_steps

        continue_scaler.scale(loss).backward()

        if step % cfg.gradient_accumulation_steps == 0 or step == len(epoch_train_loader):
            continue_scaler.unscale_(continue_optimizer)
            torch.nn.utils.clip_grad_norm_(resume_model.parameters(), cfg.max_grad_norm)
            continue_scaler.step(continue_optimizer)
            continue_scaler.update()
            continue_optimizer.zero_grad(set_to_none=True)
            continue_scheduler.step()

    val_metrics = evaluate_loader(resume_model, processor, validation_loader, device, max_new_tokens, split_name=f"more_val_{epoch}")
    test_metrics = evaluate_loader(resume_model, processor, test_loader, device, max_new_tokens, split_name=f"more_test_{epoch}")

    print(f"epoch={epoch} val_wer={val_metrics['wer']:.4f} test_wer={test_metrics['wer']:.4f}")

eval:more_val_1:   0%|          | 0/399 [00:00<?, ?it/s]

eval:more_test_1:   0%|          | 0/399 [00:00<?, ?it/s]

epoch=1 val_wer=0.2023 test_wer=0.1657


eval:more_val_2:   0%|          | 0/399 [00:00<?, ?it/s]

eval:more_test_2:   0%|          | 0/399 [00:00<?, ?it/s]

...

eval:more_test_4:   0%|          | 0/399 [00:00<?, ?it/s]

epoch=4 val_wer=0.2023 test_wer=0.1657


eval:more_val_5:   0%|          | 0/399 [00:00<?, ?it/s]

eval:more_test_5:   0%|          | 0/399 [00:00<?, ?it/s]

epoch=5 val_wer=0.2023 test_wer=0.1657


In [13]:
import random
import torch
from pprint import pprint

# 1) New generation length
max_new_tokens = 238

# 2) Choose model and dataset
model_to_use = (
    best_model if "best_model" in globals()
    else continue_model if "continue_model" in globals()
    else model
)
dataset_to_use = test_dataset

# 3) 10 random predictions
rng = random.Random()  # different samples each run
indices = rng.sample(range(len(dataset_to_use)), k=min(10, len(dataset_to_use)))

samples = []
model_to_use.eval()

for idx in indices:
    item = dataset_to_use[idx]
    batch = WhisperDataCollator(processor)([item])

    input_features = batch["input_features"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        generated_ids = model_to_use.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            language=cfg.language,
            task=cfg.task,
        )

    prediction = processor.tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    samples.append({
        "path": batch["paths"][0],
        "reference": batch["texts"][0],
        "prediction": normalize_transcription(prediction),
    })

pprint(samples)

[{'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cv_bg_train_00000_of_00001_0002984.wav',
  'prediction': 'днес когато даваше плесницата кандов почувствува че се намери '
                'в странно положение',
  'reference': 'днес когато даваше плесницата кандов почувствува че се намери '
               'в странно положение'},
 {'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_validation_00000_of_00001_0000291.wav',
  'prediction': 'писмото беше пристигнало вчера по неизвестен канал',
  'reference': 'писмото беше пристигнало вчера по неизвестен канал'},
 {'path': '/home/anna/python/MIPT/speach_recognition/FP/data/prepared_data/prepared_audio/cc0_bg_train_00003_of_00005_0000290.wav',
  'prediction': 'сиките наричат религията си гурмат което на панджавския '
                'означава пътят на гуру гуруто е фундаментален аспект във '
                'всички индийски религи но сикхизма е предобил значени